# Monthly Streamflow (`Q`) Wrangling for HUC8 Water Balance

This notebook prepares the monthly runoff / streamflow component (`Q`) for a HUC8 basin water balance workflow.

The streamflow data are stored by USGS gauge site number (`site_no`), while the basin metadata file identifies which gauges represent inflow (`gauge_in`) and outflow (`gauge_out`) for each HUC8 basin. Some HUC8 basins are also merged into larger analysis units using `merge_id`.

The purpose of this notebook is to:

1. Load the selected basin metadata and monthly streamflow summary data.
2. Create a standardized basin grouping field:
   - If a basin has a `merge_id`, it is grouped with other HUC8s that share that same `merge_id`.
   - If a basin does not have a `merge_id`, it remains its own analysis basin.
3. Build basin group metadata that catalogs which HUC8s belong to each analysis basin and sums their areas.
4. Convert the `gauge_in` and `gauge_out` columns into a long-format gauge lookup table.
5. Standardize gauge IDs so that values like `US09527700`, `09527700`, and `9527700` can be matched consistently.
6. Match monthly streamflow records to selected basin gauges using `site_no`.
7. Assign each matched streamflow record:
   - `HUCID`
   - `basin_group_id`
   - `role`, where role is either `in` or `out`
8. Identify and exclude internal gauges that occur within merged basin groups, so internal transfers are not counted as external basin inflow or outflow.
9. Summarize monthly streamflow by basin group into:
   - `q_in_cfs`
   - `q_out_cfs`
   - `q_stream_balance_cfs = q_in_cfs - q_out_cfs`
   - `q_runoff_cfs = q_out_cfs - q_in_cfs`

The resulting monthly `Q` table can be joined later with precipitation (`P`) and evapotranspiration (`ET`) data to support either equivalent water balance form:

`ΔS = P - ET + Q_stream_balance`

or:

`ΔS = P - ET - Q_runoff`

where `Q_stream_balance` is positive when streamflow input exceeds streamflow output, and `Q_runoff` is positive when streamflow export from the basin exceeds inflow.


In [8]:
# Load Libraries 
import pandas as pd
import numpy as np
from pathlib import Path


data_dir = Path("data") # set data directory path

selected_path = data_dir / "metadata" / "Selected_basins.csv"
q_path = data_dir / "q" / "all_gauges_monthly_streamflow_summary.csv"
et_path = data_dir / "et" / "openet_monthly_huc8_ensemble_primary_2000_2025.csv"

metadata_dir = data_dir / "metadata"
q_dir = data_dir / "q"


## 2. Load selected basin metadata and define basin groups.

This block loads the Selected_basins.csv file, cleans the key fields, and creates the final analysis unit called basin_group_id.

In [9]:
selected = pd.read_csv(selected_path)

selected.columns = selected.columns.str.strip() # removes leading/trailing whitespace. 

# Clean up the HUCIDs all as clean 8-character strings. 
selected["HUCID"] = (
    selected["HUCID"]
    .astype(str)        # convert to string
    .str.strip()        # remove leading/trailing whitespace
    .str.replace(r"\.0$", "", regex=True) # remove trailing .0 if present. 
    .str.zfill(8)       # pad the values with leading zeros E.g. 3060101 --> 03060101
)

# Matters because HUC IDs can begin with zero, and losing that zero would break matching.

# Convert merge_id to numeric, coercing errors to NaN. 
selected["merge_id"] = pd.to_numeric(selected["merge_id"], errors="coerce")

# So, now merge_id is either: 
# merge_id = 1, 2, 3, etc.  → merged basin group
# merge_id = blank / NaN    → individual HUC basin

# Final analysis unit:
# if merge_id exists, use merge_id group;
# otherwise use individual HUCID
selected["basin_group_id"] = np.where(   # If
    selected["merge_id"].notna(),        # selected merge_id is not blank, merge. 
    "merge_" + selected["merge_id"].astype("Int64").astype(str), # otherwise use the HUCID
    selected["HUCID"]
)

selected.head()

,HUCID,gauge_in,gauge_out,confidence,notes,decisions,merge_id,basin_group_id
0,18100204,"US09527700, US10259540",NaN,high,"endorheic basin, no outflow",inc,NaN,18100204
1,18090202,NaN,US10251330,high,NaN,inc,NaN,18090202
2,18070203,NaN,US11078000,high,add 70202,inc,6.0,merge_6
3,18070202,NaN,US11078000,high,merge with 70203,inc,6.0,merge_6
4,18060005,NaN,US11152500,high,NaN,inc,1.0,merge_1


In [10]:
# --------------------------------------------------
# Build basin group metadata with HUC membership and area
# --------------------------------------------------

et = pd.read_csv(et_path)

et.columns = et.columns.str.strip()

# Standardize HUC8 formatting in ET data
et["huc8"] = (
    et["huc8"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

# Extract one representative area record per individual HUC8.
# The ET file is monthly, so each HUC8 appears many times with the same area.
# We use mean here only to collapse repeated monthly copies of the same HUC8 area.
# Merged basin areas are summed later by basin_group_id.
et_area_by_huc = (
    et
    .groupby("huc8", as_index=False)
    .agg(
        area_acres=("areaacres", "mean"),
        area_sqkm=("areasqkm", "mean"),
        huc8_name=("name", "first")
    )
)

# Optional QA: confirm that area is constant across monthly ET records for each HUC8.
area_check = (
    et
    .groupby("huc8", as_index=False)
    .agg(
        areaacres_min=("areaacres", "min"),
        areaacres_max=("areaacres", "max"),
        areasqkm_min=("areasqkm", "min"),
        areasqkm_max=("areasqkm", "max")
    )
)

area_variation = area_check[
    (area_check["areaacres_min"] != area_check["areaacres_max"]) |
    (area_check["areasqkm_min"] != area_check["areasqkm_max"])
]

print(f"HUC8s with varying area across ET months: {len(area_variation)}")
display(area_variation.head())

# HUC-to-basin-group lookup from Selected_basins.csv
huc_to_group = (
    selected[["HUCID", "basin_group_id", "merge_id"]]
    .drop_duplicates()
    .copy()
)

# Join area onto selected HUCs
huc_to_group_area = huc_to_group.merge(
    et_area_by_huc,
    left_on="HUCID",
    right_on="huc8",
    how="left"
)

# QA: check whether any selected HUCs are missing area
missing_area = huc_to_group_area[huc_to_group_area["area_acres"].isna()]

print(f"Selected HUCs missing ET area values: {len(missing_area)}")
display(missing_area)


HUC8s with varying area across ET months: 0


,huc8,areaacres_min,areaacres_max,areasqkm_min,areasqkm_max


Selected HUCs missing ET area values: 1


,HUCID,basin_group_id,merge_id,huc8,area_acres,area_sqkm,huc8_name
13,18010212,merge_2,2.0,NaN,NaN,NaN,NaN


In [11]:
def first_nonnull(series):
    values = series.dropna()
    if len(values) == 0:
        return pd.NA
    return values.iloc[0]


basin_group_metadata = (
    huc_to_group_area
    .groupby("basin_group_id", as_index=False)
    .agg(
        hucids_in_group=("HUCID", lambda x: ", ".join(sorted(set(x)))),
        n_huc8=("HUCID", "nunique"),
        merge_id=("merge_id", first_nonnull),
        area_acres_total=("area_acres", lambda x: x.sum(min_count=1)),
        area_sqkm_total=("area_sqkm", lambda x: x.sum(min_count=1))
    )
)

basin_group_metadata["is_merged"] = basin_group_metadata["n_huc8"] > 1

basin_group_metadata = basin_group_metadata[
    [
        "basin_group_id",
        "is_merged",
        "merge_id",
        "n_huc8",
        "hucids_in_group",
        "area_acres_total",
        "area_sqkm_total"
    ]
]

basin_group_metadata = basin_group_metadata.sort_values("basin_group_id")

display(basin_group_metadata.head(20))


,basin_group_id,is_merged,merge_id,n_huc8,hucids_in_group,area_acres_total,area_sqkm_total
0,17010102,False,<NA>,1,17010102,539423.88,2182.97
1,17010201,False,<NA>,1,17010201,1200001.78,4856.24
2,17010202,False,<NA>,1,17010202,1164572.73,4712.86
3,17010203,False,<NA>,1,17010203,1480179.81,5990.08
4,17010205,False,<NA>,1,17010205,1828999.92,7401.71
5,17010207,False,<NA>,1,17010207,726348.68,2939.43
6,17010209,False,<NA>,1,17010209,1072564.75,4340.52
7,17010211,False,<NA>,1,17010211,466558.53,1888.10
8,17010212,False,<NA>,1,17010212,1285641.46,5202.81
9,17010301,False,<NA>,1,17010301,573785.43,2322.03


In [12]:
basin_group_metadata.to_csv(
    metadata_dir / "basin_group_metadata_with_area.csv",
    index=False
)


## 3. Define helper functions for cleaning gauge IDs
This block defines two reusable functions:
normalize_site_no()
split_gage_cell()

In [13]:
def normalize_site_no(value):
    """
    Convert USGS-style site numbers to 8-digit strings.

    Examples:
    'US09527700' -> '09527700'
    9527700      -> '09527700'
    '09527700'   -> '09527700'
    """
    if pd.isna(value): # If the value is null
        return pd.NA   # Return null for null values. 
    
    digits = "".join(ch for ch in str(value) if ch.isdigit()) # Extract only the numeric characters
    
    if digits == "": # If no numeric characters found
        return pd.NA # Return null for empty numeric strings
    
    return digits.zfill(8) # Pad with leading zeros to make it 8 characters long


def split_gage_cell(value): # Split cells that may contain multiple gauges separated by commas.
 
    if pd.isna(value): # If the value is null
        return []      # Return an empty list for null values
    
    return [ # Return a list of stripped gauge names
        g.strip() # Strip whitespace from each gauge name
        for g in str(value).split(",") # Split the value by commas
        if g.strip() # Only include non-empty gauge names
    ]

The purpose of this block is to transform the selected basin table from this kind of structure:
```
HUCID      basin_group_id   gauge_in      gauge_out
18060004   merge_1          US11111111    US22222222
18060005   merge_1                       US33333333

# into this kind of structure
HUCID      basin_group_id   site_no_raw   site_no    role
18060004   merge_1          US11111111    11111111   in
18060004   merge_1          US22222222    22222222   out
18060005   merge_1          US33333333    33333333   out

```


In [14]:
gage_rows = [] # Initialize an empty list to store the gauge rows

for _, row in selected.iterrows(): # Iterate over each row in the selected DataFrame
    hucid = row["HUCID"] # Extract the HUCID from the current row
    basin_group_id = row["basin_group_id"] # Extract the basin_group_id from the current row

    # Inflow gauges
    for raw_gage in split_gage_cell(row["gauge_in"]): # Iterate over each raw gauge in the inflow column
        gage_rows.append({ # Append a dictionary for each gauge to the gage_rows list
            "HUCID": hucid, # The HUCID for the current row
            "basin_group_id": basin_group_id, # The basin_group_id for the current row
            "site_no_raw": raw_gage, # The raw site number for the current row
            "site_no": normalize_site_no(raw_gage), # The normalized site number for the current row
            "role": "in" # The role for the current row
        })

    # Outflow gauges
    for raw_gage in split_gage_cell(row["gauge_out"]): # Iterate over each raw gauge in the outflow column
        gage_rows.append({ # Append a dictionary for each gauge to the gage_rows list
            "HUCID": hucid, # The HUCID for the current row
            "basin_group_id": basin_group_id, # The basin_group_id for the current row
            "site_no_raw": raw_gage, # The raw site number for the current row
            "site_no": normalize_site_no(raw_gage), # The normalized site number for the current row
            "role": "out" # The role for the current row
        })


gage_lookup = pd.DataFrame(gage_rows) # Create a DataFrame from the list of gauge rows

gage_lookup = ( # Create a DataFrame from the list of gauge rows
    gage_lookup 
    .dropna(subset=["site_no"]) # Drop rows where site_no is null
    .drop_duplicates(subset=["basin_group_id", "HUCID", "site_no", "role"]) # Drop duplicate rows
    .sort_values(["basin_group_id", "HUCID", "role", "site_no"]) # Sort the DataFrame
    .reset_index(drop=True) # Reset the index
)

gage_lookup.head(20) # Display the first 20 rows of the gage_lookup DataFrame

,HUCID,basin_group_id,site_no_raw,site_no,role
0,17010102,17010102,US12302055,12302055,out
1,17010201,17010201,US12324680,12324680,out
2,17010202,17010202,US12324680,12324680,in
3,17010202,17010202,US12334550,12334550,out
4,17010203,17010203,US12372000,12372000,in
5,17010203,17010203,US12340000,12340000,out
6,17010205,17010205,US12352500,12352500,out
7,17010207,17010207,US12358500,12358500,out
8,17010209,17010209,US12362500,12362500,out
9,17010211,17010211,US12370000,12370000,out


## 5. Identify internal gauges within merged basin groups
This block detects gauges that should not be counted as external basin inflow or outflow after HUCs have been merged.

The issue is this: a gauge may be the outflow gauge for one HUC, but the inflow gauge for another HUC in the same merged basin group. Once those HUCs are merged, that gauge is no longer a boundary of the analysis basin. It is an internal transfer point.

For a merged group, we want to count only true external boundary fluxes:

external inflow  → water entering the merged basin group
external outflow → water leaving the merged basin group
internal gauge   → water moving between HUCs inside the merged group

In [15]:
role_check = ( # Create a grouped DataFrame to check the roles of each gauge
    gage_lookup
    .groupby(["basin_group_id", "site_no"], as_index=False) # Group by basin_group_id and site_no
    .agg( # Aggregate the grouped data
        roles=("role", lambda x: ",".join(sorted(set(x)))), # Join the unique roles for each group
        n_roles=("role", "nunique"), # Count the unique roles for each group. How many different roles does a guage have? 
        hucids=("HUCID", lambda x: ",".join(sorted(set(x)))) 
        # Join the unique HUCIDs for each group. lists which HUCIDs are associated 
        # with that gauge inside a basin group. 
        # USeful for checking which HUCs share an internal gauge. 
    )
)

# Creates a three column dataframe summary. If a gauge appears as both an in and out
# then it's probably an internal gauge within a merged basin unit. 

internal_gages = role_check.loc[role_check["n_roles"] > 1, ["basin_group_id", "site_no"]]
# Filter the role_check DataFrame to include only rows where n_roles is greater than 1
# They get treated as internal gauges. 

# Add a new column called boundary status to the gauge lookup DataFrame. 
gage_lookup = gage_lookup.merge(
    internal_gages.assign(boundary_status="internal"), # Assign "internal" status to filtered gauges.
    on=["basin_group_id", "site_no"], # Merge on basin_group_id and site_no
    how="left" # Perform a left join
)

gage_lookup["boundary_status"] = gage_lookup["boundary_status"].fillna("external")
# Any gauge that was not marked as internal is now labeled "external"

gage_lookup.sort_values(["basin_group_id", "site_no", "role"]).head(30)
# Display the first 30 rows of the updated gauge lookup DataFrame. 

,HUCID,basin_group_id,site_no_raw,site_no,role,boundary_status
0,17010102,17010102,US12302055,12302055,out,external
1,17010201,17010201,US12324680,12324680,out,external
2,17010202,17010202,US12324680,12324680,in,external
3,17010202,17010202,US12334550,12334550,out,external
5,17010203,17010203,US12340000,12340000,out,external
4,17010203,17010203,US12372000,12372000,in,external
6,17010205,17010205,US12352500,12352500,out,external
7,17010207,17010207,US12358500,12358500,out,external
8,17010209,17010209,US12362500,12362500,out,external
9,17010211,17010211,US12370000,12370000,out,external


In [16]:
# Make a copy of the gauge lookup DataFrame containing only external gauges. 
external_gage_lookup = gage_lookup[gage_lookup["boundary_status"] == "external"].copy()

In [17]:
gage_lookup.to_csv(
    metadata_dir / "gage_lookup_from_selected_basins.csv",
    index=False
) # Save the gage_lookup DataFrame to a CSV file


In [18]:
q = pd.read_csv(q_path) # read the runoff data. 

q.columns = q.columns.str.strip() # Remove leading/trailing whitespace from column names

q["site_no"] = q["site_no"].apply(normalize_site_no) # Normalize the site numbers
q["year_month"] = pd.to_datetime(q["year_month"]) # Convert year_month to datetime

q.head() # Display the first 5 rows of the DataFrame

,site_no,year_month,water_year,year,month,mean_discharge_cfs,sum_daily_mean_cfs,n_days,n_missing_values
0,09527700,2011-10-01,2012,2011,10,3350.000000,20100.0,6,0
1,09527700,2011-11-01,2012,2011,11,2551.000000,76530.0,30,0
2,09527700,2011-12-01,2012,2011,12,1904.451613,59038.0,31,0
3,09527700,2012-01-01,2012,2012,1,2339.548387,72526.0,31,0
4,09527700,2012-02-01,2012,2012,2,2935.862069,85140.0,29,0


In [19]:
q_matched = q.merge( # Merge the runoff data with the external gauge lookup
    external_gage_lookup[ # 
        ["HUCID", "basin_group_id", "site_no", "role", "boundary_status"]
    ],
    on="site_no", # Merge on the site_no column
    how="inner" # Perform an inner join
)

q_matched.head() # Display the first 5 rows of the matched DataFrame

,site_no,year_month,water_year,year,month,mean_discharge_cfs,sum_daily_mean_cfs,n_days,n_missing_values,HUCID,basin_group_id,role,boundary_status
0,09527700,2011-10-01,2012,2011,10,3350.000000,20100.0,6,0,18100204,18100204,in,external
1,09527700,2011-11-01,2012,2011,11,2551.000000,76530.0,30,0,18100204,18100204,in,external
2,09527700,2011-12-01,2012,2011,12,1904.451613,59038.0,31,0,18100204,18100204,in,external
3,09527700,2012-01-01,2012,2012,1,2339.548387,72526.0,31,0,18100204,18100204,in,external
4,09527700,2012-02-01,2012,2012,2,2935.862069,85140.0,29,0,18100204,18100204,in,external


In [20]:
q_matched.to_csv(q_dir / "q_matched_to_selected_basins.csv", index=False) 
# Save the matched DataFrame to a CSV file


In [21]:
q_by_role = (   # Group the matched DataFrame by basin group ID and time periods
    q_matched 
    .groupby( # Group by the specified columns
        ["basin_group_id", "year_month", "water_year", "year", "month", "role"],
        as_index=False
    )
    .agg( # Aggregate the grouped data
        q_cfs_total=("mean_discharge_cfs", "sum"), # Sum simultaneous mean discharge across boundary gauges
        n_gages=("site_no", "nunique"), # Number of unique gages
        site_nos=("site_no", lambda x: ",".join(sorted(set(x)))), # Comma-separated list of site numbers
        n_missing_values=("n_missing_values", "sum") # Sum of missing values
    )
)

q_by_role.head()


,basin_group_id,year_month,water_year,year,month,role,q_cfs_total,n_gages,site_nos,n_missing_values
0,17010102,2000-01-01,2000,2000,1,out,170.129032,1,12302055,0
1,17010102,2000-02-01,2000,2000,2,out,221.655172,1,12302055,0
2,17010102,2000-03-01,2000,2000,3,out,427.806452,1,12302055,0
3,17010102,2000-04-01,2000,2000,4,out,1617.866667,1,12302055,0
4,17010102,2000-05-01,2000,2000,5,out,1123.806452,1,12302055,0


In [22]:
q_monthly = ( # Create a pivot table to reshape the data
    q_by_role 
    .pivot_table( # Create a pivot table
        index=["basin_group_id", "year_month", "water_year", "year", "month"], # Index columns for the pivot table
        columns="role", # Column to use for creating the pivot table
        values="q_cfs_total",
        aggfunc="sum"
    )
    .reset_index() # Reset the index to make the columns regular
)

q_monthly.columns.name = None # make column names regular. 

if "in" not in q_monthly.columns: # Check if the "in" column exists
    q_monthly["in"] = 0.0

if "out" not in q_monthly.columns: # Check if the "out" column exists
    q_monthly["out"] = 0.0

q_monthly = q_monthly.rename(columns={ # Rename the columns
    "in": "q_in_cfs",
    "out": "q_out_cfs"
})

q_monthly["q_in_cfs"] = q_monthly["q_in_cfs"].fillna(0) # Fill missing values with 0
q_monthly["q_out_cfs"] = q_monthly["q_out_cfs"].fillna(0) # Fill missing values with 0

# Net streamflow balance:
# positive = net inflow to basin; negative = net export/runoff from basin
q_monthly["q_stream_balance_cfs"] = q_monthly["q_in_cfs"] - q_monthly["q_out_cfs"]

# Net runoff/export from the analysis basin:
# positive = more streamflow leaves the basin than enters it
q_monthly["q_runoff_cfs"] = q_monthly["q_out_cfs"] - q_monthly["q_in_cfs"]

q_monthly = q_monthly.sort_values(["basin_group_id", "year_month"])

q_monthly.to_csv(q_dir / "q_monthly_by_basin_group.csv", index=False)

q_monthly.head()


,basin_group_id,year_month,water_year,year,month,q_in_cfs,q_out_cfs,q_stream_balance_cfs,q_runoff_cfs
0,17010102,2000-01-01,2000,2000,1,0.0,170.129032,-170.129032,170.129032
1,17010102,2000-02-01,2000,2000,2,0.0,221.655172,-221.655172,221.655172
2,17010102,2000-03-01,2000,2000,3,0.0,427.806452,-427.806452,427.806452
3,17010102,2000-04-01,2000,2000,4,0.0,1617.866667,-1617.866667,1617.866667
4,17010102,2000-05-01,2000,2000,5,0.0,1123.806452,-1123.806452,1123.806452


In [23]:
q_monthly_with_area = q_monthly.merge(
    basin_group_metadata[
        [
            "basin_group_id",
            "is_merged",
            "n_huc8",
            "hucids_in_group",
            "area_acres_total",
            "area_sqkm_total"
        ]
    ],
    on="basin_group_id",
    how="left"
)

display(q_monthly_with_area.head())

q_monthly_with_area.to_csv(
    q_dir / "q_monthly_by_basin_group_with_area.csv",
    index=False
)


,basin_group_id,year_month,water_year,year,month,q_in_cfs,q_out_cfs,q_stream_balance_cfs,q_runoff_cfs,is_merged,n_huc8,hucids_in_group,area_acres_total,area_sqkm_total
0,17010102,2000-01-01,2000,2000,1,0.0,170.129032,-170.129032,170.129032,False,1,17010102,539423.88,2182.97
1,17010102,2000-02-01,2000,2000,2,0.0,221.655172,-221.655172,221.655172,False,1,17010102,539423.88,2182.97
2,17010102,2000-03-01,2000,2000,3,0.0,427.806452,-427.806452,427.806452,False,1,17010102,539423.88,2182.97
3,17010102,2000-04-01,2000,2000,4,0.0,1617.866667,-1617.866667,1617.866667,False,1,17010102,539423.88,2182.97
4,17010102,2000-05-01,2000,2000,5,0.0,1123.806452,-1123.806452,1123.806452,False,1,17010102,539423.88,2182.97


In [24]:
# Which selected gauges did not find streamflow records?
selected_gages = set(gage_lookup["site_no"]) # Get the set of selected gage IDs
q_gages = set(q["site_no"])

missing_from_q = (
    gage_lookup # Filter the gage lookup DataFrame
    .loc[~gage_lookup["site_no"].isin(q_gages)]
    .sort_values(["basin_group_id", "HUCID", "role", "site_no"])
)

missing_from_q# Display the missing gages. 

,HUCID,basin_group_id,site_no_raw,site_no,role,boundary_status
49,17080002,17080002,US14222500,14222500,out,external


In [25]:
# Which streamflow gauges were matched?
matched_gages = ( # Create a DataFrame of matched gages
    q_matched[["basin_group_id", "HUCID", "site_no", "role"]] # Select the relevant columns
    .drop_duplicates() # Remove duplicate rows
    .sort_values(["basin_group_id", "HUCID", "role", "site_no"]) # Sort the DataFrame
)

matched_gages.head(50)

,basin_group_id,HUCID,site_no,role
7769,17010102,17010102,12302055,out
8081,17010201,17010201,12324680,out
8082,17010202,17010202,12324680,in
8705,17010202,17010202,12334550,out
10577,17010203,17010203,12372000,in
9017,17010203,17010203,12340000,out
9329,17010205,17010205,12352500,out
9641,17010207,17010207,12358500,out
9953,17010209,17010209,12362500,out
10265,17010211,17010211,12370000,out


In [26]:
# Internal gauges created by merging
role_check[role_check["n_roles"] > 1] # Filter for gauges with more than one role
# If it returns an empty DataFrame it means: 
# No gauges are shared as both in/out within the same merged group
# The merge/gauge relationships are not being detected correctly

# Rows returned = good, internal gauges found and can be excluded from final Q
# No rows returned = maybe okay, but worth checking whether merged basins truly share gauges


,basin_group_id,site_no,roles,n_roles,hucids
